In [ ]:
import torch
from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    Trainer,
    TrainingArguments
)
from datasets import load_from_disk
import numpy as np

In [ ]:
MODEL_NAME = "google/mt5-small"

TRAIN_PATH = "../data/processed/train_dataset"
VAL_PATH = "../data/processed/val_dataset"

OUTPUT_DIR = "../models/mt5-haoussa-zarma"

In [ ]:
train_dataset = load_from_disk(TRAIN_PATH)
val_dataset = load_from_disk(VAL_PATH)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

In [ ]:
model = MT5ForConditionalGeneration.from_pretrained(MODEL_NAME)
tokenizer = MT5Tokenizer.from_pretrained(MODEL_NAME)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model.to(device)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    result = bleu.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": result["score"]}

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_dir="../logs",
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # accélération GPU
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
def translate(text):
    input_text = "translate Hausa to Zarma: " + text

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_length=32)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(translate("Sannu"))
print(translate("Ina gidan ku?"))

In [ ]:
"""
Observations:

1. Le modèle converge progressivement
2. BLEU score utilisé pour évaluer qualité traduction
3. Dataset limité → apprentissage rapide mais risque d’overfitting
4. mT5 adapté aux langues multiples mais nécessite fine-tuning

Conclusion:
Le modèle apprend des correspondances basiques Hausa → Zarma.
"""